# Aula 14 - Algoritmo de Dijkstra e Roteamento Dinâmico em Tempo Real (H2)

Neste notebook implementamos o **Algoritmo de Dijkstra** utilizando fila de prioridade binária (`heapq`) para roteamento ótimo (menor perda de carga ou distância) de hidrogênio.

In [1]:
import heapq
from typing import List, Tuple, Optional, Set, Dict, Any

def formatar_tabela(dados):
    if not dados: return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

class GrafoTubulacao:
    def __init__(self, vertices):
        self.vertices = vertices
        self.v_to_idx = {v: i for i, v in enumerate(vertices)}
        self.idx_to_v = {i: v for i, v in enumerate(vertices)}
        self.n = len(vertices)
        self.adj_binaria = [[0] * self.n for _ in range(self.n)]
        self.adj_pesos = [[float('inf')] * self.n for _ in range(self.n)]
        for i in range(self.n): self.adj_pesos[i][i] = 0.0
        self.arestas_detalhes = []

    def adicionar_tubulacao(self, origem, destino, comprimento_m, tag_valvula, diametro_pol=1.0):
        u = self.v_to_idx[origem]
        v = self.v_to_idx[destino]
        self.adj_binaria[u][v] = 1
        self.adj_pesos[u][v] = comprimento_m
        self.arestas_detalhes.append({
            "Origem": origem, "Destino": destino,
            "Comprimento (m)": comprimento_m, "Válvula ISA": tag_valvula, "Diâmetro (pol)": diametro_pol
        })

def criar_rede_estacao_h2():
    nos = ["E-101", "C-101", "C-102", "TK-101", "TK-102", "CH-101", "MAN-101", "D-101"]
    g = GrafoTubulacao(nos)
    g.adicionar_tubulacao("E-101", "C-101", 5.0, "XV-101", 2.0)
    g.adicionar_tubulacao("C-101", "TK-101", 10.0, "XV-102", 1.0)
    g.adicionar_tubulacao("C-101", "C-102", 15.0, "XV-103", 1.0)
    g.adicionar_tubulacao("C-102", "TK-102", 8.0, "XV-104", 0.5)
    g.adicionar_tubulacao("TK-101", "MAN-101", 12.0, "XV-105", 1.0)
    g.adicionar_tubulacao("TK-102", "MAN-101", 15.0, "XV-106", 0.5)
    g.adicionar_tubulacao("CH-101", "MAN-101", 6.0, "XV-107", 1.0)
    g.adicionar_tubulacao("MAN-101", "D-101", 4.0, "XV-108", 0.5)
    return g

rede = criar_rede_estacao_h2()

class RoteadorDijkstra:
    def __init__(self, grafo: GrafoTubulacao):
        self.g = grafo

    def calcular_menor_caminho(self, origem: str, destino: str, 
                               bloqueios: Optional[Set[str]] = None) -> Tuple[float, List[str]]:
        if bloqueios is None: bloqueios = set()
        if origem in bloqueios or destino in bloqueios: return float('inf'), []
        
        dist = {v: float('inf') for v in self.g.vertices}
        pred = {v: None for v in self.g.vertices}
        dist[origem] = 0.0
        heap = [(0.0, origem)]
        
        while heap:
            d_u, u = heapq.heappop(heap)
            if d_u > dist[u]: continue
            if u == destino: break
            
            u_idx = self.g.v_to_idx[u]
            for v_idx in range(self.g.n):
                v = self.g.idx_to_v[v_idx]
                peso = self.g.adj_pesos[u_idx][v_idx]
                if peso < float('inf') and v not in bloqueios:
                    nova_d = d_u + peso
                    if nova_d < dist[v]:
                        dist[v] = nova_d
                        pred[v] = u
                        heapq.heappush(heap, (nova_d, v))
                        
        caminho = []
        atual = destino
        while atual is not None:
            caminho.append(atual)
            atual = pred[atual]
        caminho.reverse()
        if caminho and caminho[0] == origem:
            return dist[destino], caminho
        return float('inf'), []

roteador = RoteadorDijkstra(rede)
custo_nom, rota_nom = roteador.calcular_menor_caminho("E-101", "D-101")
print(f"Rota Nominal Ótima (Dijkstra): {' -> '.join(rota_nom)} | Comprimento/Perda: {custo_nom:.1f} m")
print("\nEstrutura da Rede:")
print(formatar_tabela(rede.arestas_detalhes))


Rota Nominal Ótima (Dijkstra): E-101 -> C-101 -> TK-101 -> MAN-101 -> D-101 | Comprimento/Perda: 31.0 m

Estrutura da Rede:
Origem | Destino | Comprimento (m) | Válvula ISA | Diâmetro (pol)
-------+---------+-----------------+-------------+---------------
E-101  | C-101   | 5.0             | XV-101      | 2.0           
C-101  | TK-101  | 10.0            | XV-102      | 1.0           
C-101  | C-102   | 15.0            | XV-103      | 1.0           
C-102  | TK-102  | 8.0             | XV-104      | 0.5           
TK-101 | MAN-101 | 12.0            | XV-105      | 1.0           
TK-102 | MAN-101 | 15.0            | XV-106      | 0.5           
CH-101 | MAN-101 | 6.0             | XV-107      | 1.0           
MAN-101| D-101   | 4.0             | XV-108      | 0.5           
